In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
import sys
sys.path.append("..")

In [3]:
from module1.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
len(documents)

103

In [6]:
len(ground_truth)

515

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()
from toyaikit.llm import OpenAIClient

In [8]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [11]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [12]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Is it still okay to join this course if I found it late?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late join course okay found it late join after start FAQ"}', call_id='call_VbyWyQzkBjlZTT0pY9stUHc6', name='search', type='function_call', id='fc_037c0516d0cd008d006a4c9289f14881a08d0e47c8e3892b34', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_VbyWyQzkBjlZTT0pY9stUHc6',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re s

In [13]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [14]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late join course okay found it late join after start FAQ"}'}]

In [15]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [16]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

100%|██████████| 50/50 [00:21<00:00,  2.38it/s]


In [17]:
df_agent = pd.DataFrame(agent_answers)

In [19]:
pd.set_option("display.max_colwidth", None)

In [20]:
df_agent.head(1)

,question,answer_agent,answer_orig,tool_calls,cost,document
0,Is it still okay to join this course if I found it late?,"Yes — you can still join if you discovered the course late. The course materials are available, and you can start whenever you want.\n\nIf you want a certificate, make sure you submit the project while submissions are still being accepted.","Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.","[{'name': 'search', 'arguments': '{""query"":""join course late found it late okay late enrollment""}'}]",0.0010245,74eb249bbf


In [21]:
df_agent["cost"].sum()

Decimal('0.06209550')

In [22]:
agent_answers = df_agent.to_dict(orient="records")

In [23]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [24]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [25]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

In [26]:
def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [27]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It says it is okay to join the course late and includes the key condition for receiving a certificate: the project must be submitted while submissions are still being accepted.', answer_score='good', trajectory_reasoning='The search query was relevant to the question and included important keywords like joining late and the course. Only one search was needed, and it was sufficient to support the final answer.', trajectory_score='good')

In [28]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [29]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

100%|██████████| 50/50 [00:15<00:00,  3.13it/s]


In [30]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [31]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [32]:
df_agent_eval.head(1)

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,Is it still okay to join this course if I found it late?,74eb249bbf,good,"The agent’s answer matches the ground truth. It says that it is okay to join the course late and includes the important condition that to receive a certificate, the project must be submitted while submissions are still being accepted.",good,"The single search query is relevant to the question and uses key ideas like joining late/late enrollment. One search was sufficient, and there were no unnecessary duplicate calls."


In [33]:
calc_total_price(usages)

0.05341275

In [34]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    50
Name: count, dtype: int64

In [35]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64